# ArcFace 人脸识别训练 (Kaggle 版)

IResNet50 + ArcFace, 训练 CASIA-WebFace。

**数据是原始 `.rec`**(不是图片), 需先在本 notebook 里转换。用到的公开数据集:
- 训练 `.rec`: `/kaggle/input/datasets/debarghamitraroy/casia-webface/casia-webface/train.rec` (+ train.idx)
- 验证 `.bin`: `/kaggle/input/datasets/debarghamitraroy/casia-webface/eval`

**步骤**: 依次运行代码块 → ① 转换 .rec → ② 冒烟训练确认 → ③ 正式训练。

**注意**:
- 需开启 **GPU 加速器** (右侧 Settings → Accelerator → GPU T4 x2)。
- 转换约 20-40 分钟 (49 万张); 转完可「Create Dataset」存下图片, 下次免转换。
- T4 16GB 显存, batch=256 可用; OOM 则改 128。
- Kaggle 单次 GPU 会话约 **12h 上限**, 20 epoch 可能压线。
- 训练产物在 `/kaggle/working/output/`, 会话结束不保留, 记得下载 `last.pt`。

In [ ]:
%%writefile iresnet.py
"""
============================================================
IResNet (Improved ResNet) 人脸识别主干
insightface 风格: BN→Conv→BN→PReLU→Conv→BN 基本块, 专为 112×112 设计
(保持分辨率: stem 用 3x3 stride1, 不用 7x7 stride2 + maxpool)

输出 512 维 embedding (经 BatchNorm1d 归一化)
============================================================
"""
import torch
from torch import nn

__all__ = ['iresnet18', 'iresnet34', 'iresnet50', 'iresnet100']


def conv3x3(in_planes, out_planes, stride=1, groups=1, dilation=1):
    return nn.Conv2d(in_planes, out_planes, kernel_size=3, stride=stride,
                     padding=dilation, groups=groups, bias=False, dilation=dilation)


def conv1x1(in_planes, out_planes, stride=1):
    return nn.Conv2d(in_planes, out_planes, kernel_size=1, stride=stride, bias=False)


class IBasicBlock(nn.Module):
    expansion = 1

    def __init__(self, inplanes, planes, stride=1, downsample=None,
                 groups=1, base_width=64, dilation=1):
        super(IBasicBlock, self).__init__()
        if groups != 1 or base_width != 64:
            raise ValueError('IBasicBlock only supports groups=1 and base_width=64')
        if dilation > 1:
            raise NotImplementedError("Dilation > 1 not supported in IBasicBlock")
        self.bn1 = nn.BatchNorm2d(inplanes, eps=1e-05)
        self.conv1 = conv3x3(inplanes, planes)
        self.bn2 = nn.BatchNorm2d(planes, eps=1e-05)
        self.prelu = nn.PReLU(planes)
        self.conv2 = conv3x3(planes, planes, stride)
        self.bn3 = nn.BatchNorm2d(planes, eps=1e-05)
        self.downsample = downsample
        self.stride = stride

    def forward(self, x):
        identity = x
        out = self.bn1(x)
        out = self.conv1(out)
        out = self.bn2(out)
        out = self.prelu(out)
        out = self.conv2(out)
        out = self.bn3(out)
        if self.downsample is not None:
            identity = self.downsample(x)
        out += identity
        return out


class IResNet(nn.Module):
    fc_scale = 7 * 7  # 112×112 输入 → 最后一层特征图 7×7

    def __init__(self, block, layers, dropout=0, num_features=512,
                 zero_init_residual=False, groups=1, width_per_group=64,
                 replace_stride_with_dilation=None):
        super(IResNet, self).__init__()
        self.inplanes = 64
        self.dilation = 1
        if replace_stride_with_dilation is None:
            replace_stride_with_dilation = [False, False, False]
        self.groups = groups
        self.base_width = width_per_group

        self.conv1 = nn.Conv2d(3, self.inplanes, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(self.inplanes, eps=1e-05)
        self.prelu = nn.PReLU(self.inplanes)

        self.layer1 = self._make_layer(block, 64, layers[0], stride=2)
        self.layer2 = self._make_layer(block, 128, layers[1], stride=2,
                                       dilate=replace_stride_with_dilation[0])
        self.layer3 = self._make_layer(block, 256, layers[2], stride=2,
                                       dilate=replace_stride_with_dilation[1])
        self.layer4 = self._make_layer(block, 512, layers[3], stride=2,
                                       dilate=replace_stride_with_dilation[2])

        self.bn2 = nn.BatchNorm2d(512 * block.expansion, eps=1e-05)
        self.dropout = nn.Dropout(p=dropout, inplace=True)
        self.fc = nn.Linear(512 * block.expansion * self.fc_scale, num_features)
        self.features = nn.BatchNorm1d(num_features, eps=1e-05)
        nn.init.constant_(self.features.weight, 1.0)
        self.features.weight.requires_grad = False

        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, (nn.BatchNorm2d, nn.GroupNorm)):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
        if zero_init_residual:
            for m in self.modules():
                if isinstance(m, IBasicBlock):
                    nn.init.constant_(m.bn3.weight, 0)

    def _make_layer(self, block, planes, blocks, stride=1, dilate=False):
        downsample = None
        previous_dilation = self.dilation
        if dilate:
            self.dilation *= stride
            stride = 1
        if stride != 1 or self.inplanes != planes * block.expansion:
            downsample = nn.Sequential(
                conv1x1(self.inplanes, planes * block.expansion, stride),
                nn.BatchNorm2d(planes * block.expansion, eps=1e-05),
            )
        layers = [block(self.inplanes, planes, stride, downsample,
                        self.groups, self.base_width, previous_dilation)]
        self.inplanes = planes * block.expansion
        for _ in range(1, blocks):
            layers.append(block(self.inplanes, planes, groups=self.groups,
                                base_width=self.base_width, dilation=self.dilation))
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.prelu(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.bn2(x)
        x = torch.flatten(x, 1)
        x = self.dropout(x)
        x = self.fc(x)
        x = self.features(x)
        return x


def _iresnet(arch, block, layers, **kwargs):
    return IResNet(block, layers, **kwargs)


def iresnet18(**kwargs):
    return _iresnet('iresnet18', IBasicBlock, [2, 2, 2, 2], **kwargs)


def iresnet34(**kwargs):
    return _iresnet('iresnet34', IBasicBlock, [3, 4, 6, 3], **kwargs)


def iresnet50(**kwargs):
    return _iresnet('iresnet50', IBasicBlock, [3, 4, 14, 3], **kwargs)


def iresnet100(**kwargs):
    return _iresnet('iresnet100', IBasicBlock, [3, 13, 30, 3], **kwargs)


In [ ]:
%%writefile arcface_loss.py
"""
============================================================
ArcFace 损失 + 标签平滑交叉熵
从 resnet50_reid_train/model_v2.py 复制精简版 (纯 torch, 自包含),
避免 import model_v2 时拖入 model.py 的依赖链。

ArcFaceLayer: 加性角度间隔分类器 (scale=30, margin=0.3)
LabelSmoothingCrossEntropy: 标签平滑 CE
============================================================
"""
import math

import torch
import torch.nn as nn
import torch.nn.functional as F


class ArcFaceLayer(nn.Module):
    """ArcFace 加性角度间隔分类器
    公式: logits = s * cos(θ + m)  (正确类别)
          logits = s * cos(θ)      (其他类别)
    参考: Deng et al., "ArcFace: Additive Angular Margin Loss for Deep Face Recognition"
    """

    def __init__(self, in_features, num_classes, scale=30.0, margin=0.3):
        super(ArcFaceLayer, self).__init__()
        self.in_features = in_features
        self.num_classes = num_classes
        self.scale = scale
        self.margin = margin

        self.weight = nn.Parameter(torch.FloatTensor(num_classes, in_features))
        nn.init.xavier_uniform_(self.weight)

        self.cos_m = math.cos(margin)
        self.sin_m = math.sin(margin)
        self.eps = 1e-7

    def forward(self, features, labels=None):
        """features: [B, D]  (IResNet 输出, 会被 L2 归一化)
           labels:   [B]     (None 时不做 margin, 用于推理)"""
        feat_norm = F.normalize(features, p=2, dim=1)
        w_norm = F.normalize(self.weight, p=2, dim=1)

        cos_theta = F.linear(feat_norm, w_norm)  # [B, C]

        if labels is None:
            return cos_theta * self.scale

        cos_theta = cos_theta.clamp(-1.0 + self.eps, 1.0 - self.eps)
        sin_theta = torch.sqrt(1.0 - cos_theta ** 2)
        phi = cos_theta * self.cos_m - sin_theta * self.sin_m  # cos(θ+m)

        one_hot = F.one_hot(labels, num_classes=self.num_classes).float()
        logits = one_hot * phi + (1.0 - one_hot) * cos_theta
        return logits * self.scale


class LabelSmoothingCrossEntropy(nn.Module):
    """标签平滑交叉熵: y_smooth = (1-ε)*one_hot + ε/C"""

    def __init__(self, epsilon=0.1, reduction='mean'):
        super(LabelSmoothingCrossEntropy, self).__init__()
        self.epsilon = epsilon
        self.reduction = reduction

    def forward(self, pred, target):
        n_classes = pred.size(-1)
        log_probs = F.log_softmax(pred, dim=-1)
        smooth_target = (1.0 - self.epsilon) * F.one_hot(target, n_classes).float()
        smooth_target = smooth_target + self.epsilon / n_classes
        loss = -(smooth_target * log_probs).sum(dim=-1)
        if self.reduction == 'mean':
            return loss.mean()
        elif self.reduction == 'sum':
            return loss.sum()
        return loss


In [ ]:
%%writefile kaggle_convert.py
"""
============================================================
CASIA-WebFace (.rec) → ImageFolder 图片目录 转换脚本 (Kaggle 版)
纯 Python 解析 MXNet RecordIO, 不依赖 mxnet

.rec 二进制格式 (已验证):
  .idx  文本: 每行 "key\tbyte_offset"
  .rec  每个 record: magic(4B=0xced7230a) + lencode(4B) + header(24B) + JPEG data
        header = flag(int32) + label(int32) + id(int32) + id2(float32) + 8B padding
        label 在 header[4:8] (float32 存储, 转 int), <0 为 junk(对齐失败)

输出: dataset/face_casia/images/{label}/{key}.jpg  (ImageFolder 结构)

用法:
  python kaggle_convert.py                 # 全量 (~49 万张, 约 20-40 分钟)
  python kaggle_convert.py --limit 2000    # 冒烟: 只转前 N 个 record
============================================================
"""
import os
import struct
import sys
import time

# Kaggle 公开数据集默认路径
DEFAULT_REC = '/kaggle/input/datasets/debarghamitraroy/casia-webface/casia-webface/train.rec'
DEFAULT_IDX = '/kaggle/input/datasets/debarghamitraroy/casia-webface/casia-webface/train.idx'
DEFAULT_OUT = '/kaggle/working/dataset/face_casia/images'

MAGIC = 0xced7230a
HEADER_LEN = 24  # flag(int32) + label(int32) + id(int32) + id2(float32) + 8B padding


def arg(name, default):
    try:
        i = sys.argv.index(name)
        return sys.argv[i + 1]
    except (ValueError, IndexError):
        return default


def main():
    REC = arg('--rec', DEFAULT_REC)
    IDX = arg('--idx', DEFAULT_IDX)
    OUT = arg('--out', DEFAULT_OUT)
    limit = int(arg('--limit', 0))

    # 读 .idx 文本 → [(key, offset)]
    offsets = []
    with open(IDX, 'r') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            k, v = line.split('\t')
            offsets.append((int(k), int(v)))
    print(f'[idx] {len(offsets)} 个 record')

    if limit:
        offsets = offsets[:limit]
        print(f'[limit] 只处理前 {limit} 个 record')

    os.makedirs(OUT, exist_ok=True)

    n_written = 0
    n_junk = 0
    n_bad = 0
    labels = set()          # 既计数唯一 label, 也缓存已建目录 (避免每张图都 makedirs)
    t0 = time.time()

    f = open(REC, 'rb')
    for i, (key, off) in enumerate(offsets):
        f.seek(off)
        magic = struct.unpack('<I', f.read(4))[0]
        lencode = struct.unpack('<I', f.read(4))[0]
        if magic != MAGIC:
            n_bad += 1
            continue
        header = f.read(HEADER_LEN)
        # label 实际以 float32 存储 (0x3F800000 = 1.0), 用 '<f' 读再转 int
        label_f = struct.unpack('<f', header[4:8])[0]
        data_len = lencode - HEADER_LEN
        data = f.read(data_len)

        if label_f != label_f or label_f < 0:  # NaN 或负数 = junk
            n_junk += 1
            continue
        label = int(label_f)

        if data[:2] != b'\xff\xd8':  # 非 JPEG (尾部有 ~7047 个 8 字节浮点 junk record)
            n_bad += 1
            continue

        label_dir = os.path.join(OUT, str(label))
        if label not in labels:
            os.makedirs(label_dir, exist_ok=True)
            labels.add(label)
        with open(os.path.join(label_dir, f'{key}.jpg'), 'wb') as imgf:
            imgf.write(data)
        n_written += 1

        if (i + 1) % 50000 == 0:
            el = time.time() - t0
            print(f'  ... {i+1}/{len(offsets)} 处理中, {n_written} 张已写, {el:.1f}s')

    f.close()
    el = time.time() - t0
    print(f'\n[完成] 处理 {len(offsets)} 个 record, 耗时 {el:.1f}s')
    print(f'  写入图片: {n_written}')
    print(f'  junk(对齐失败,已跳过): {n_junk}')
    print(f'  坏 record(magic 不符/非JPEG): {n_bad}')
    print(f'  身份数(唯一 label): {len(labels)}')
    print(f'  输出目录: {OUT}')


if __name__ == '__main__':
    main()


In [ ]:
%%writefile kaggle_train.py
"""
============================================================
ArcFace 人脸识别训练 (Kaggle 版, 可移植, 无 Windows 硬编码路径)
IResNet50 + ArcFace, 数据集 CASIA-WebFace

Kaggle 上数据是原始 .rec (train.rec + train.idx), 需先转换:
  kaggle_convert.py 把 .rec → /kaggle/working/dataset/face_casia/images (ImageFolder)
  验证 .bin 在 /kaggle/input/datasets/debarghamitraroy/casia-webface/eval

iresnet.py / arcface_loss.py 与本脚本同目录 (notebook 里用 %%writefile 生成)

用法:
  python kaggle_train.py --epochs 20 --batch 256
  python kaggle_train.py --epochs 1 --batch 64 --max-steps 3   # 冒烟

性能: torch.backends.cudnn.benchmark=True (固定 112×112 下 autotune,
      否则 conv 反向 dgrad 慢 4 倍)
============================================================
"""
import os
import sys
import math
import time
from datetime import datetime

SCRIPT_DIR = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, SCRIPT_DIR)

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms, datasets
from torch.amp import autocast, GradScaler
from iresnet import iresnet50
from arcface_loss import ArcFaceLayer, LabelSmoothingCrossEntropy

# 转换后的 ImageFolder 输出目录 (kaggle_convert.py 的默认输出)
DEFAULT_DATA = '/kaggle/working/dataset/face_casia/images'


def arg(name, default):
    try:
        i = sys.argv.index(name)
        return sys.argv[i + 1]
    except (ValueError, IndexError):
        return default


def main():
    # 固定输入尺寸(112×112)下让 cuDNN autotune 选最优卷积算法
    torch.backends.cudnn.benchmark = True

    EPOCHS = int(arg('--epochs', 20))
    BATCH = int(arg('--batch', 256))     # T4 16GB 可用 256; OOM 则改 128
    WORKERS = int(arg('--workers', 2))   # Kaggle 一般 4 核 CPU
    LR0 = float(arg('--lr0', 0.1))
    WARMUP = int(arg('--warmup', 2))
    DATA = arg('--data', DEFAULT_DATA)
    OUT_DIR = arg('--out', os.path.join(SCRIPT_DIR, 'output'))
    RESUME = arg('--resume', '')
    MAX_STEPS = int(arg('--max-steps', 0))  # >0 时每 epoch 只跑 N 个 batch (冒烟)

    os.makedirs(OUT_DIR, exist_ok=True)

    print('=' * 64)
    print(f'  ArcFace Face Recognition Training (Kaggle)')
    print(f'  Start: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
    print('=' * 64)
    if torch.cuda.is_available():
        gpu = torch.cuda.get_device_name(0)
        vram = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f'  GPU: {gpu} ({vram:.1f} GB VRAM)  CUDA {torch.version.cuda}  '
              f'cudnn.benchmark={torch.backends.cudnn.benchmark}')
    else:
        print('  [ERROR] CUDA not available'); sys.exit(1)

    # ---------------- 数据 ----------------
    transform = transforms.Compose([
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
    ])
    dataset = datasets.ImageFolder(DATA, transform=transform)
    num_classes = len(dataset.classes)
    loader = DataLoader(dataset, batch_size=BATCH, shuffle=True,
                        num_workers=WORKERS, pin_memory=True, drop_last=True,
                        persistent_workers=(WORKERS > 0))
    iters_per_epoch = len(loader)
    print(f'  数据: {DATA}')
    print(f'  样本数: {len(dataset)}  身份数: {num_classes}  iters/epoch: {iters_per_epoch}')
    print(f'  配置: epochs={EPOCHS} batch={BATCH} workers={WORKERS} lr0={LR0} warmup={WARMUP}')

    # ---------------- 模型 ----------------
    backbone = iresnet50(num_features=512).cuda()
    arcface = ArcFaceLayer(512, num_classes, scale=30.0, margin=0.3).cuda()
    ce_loss = LabelSmoothingCrossEntropy(epsilon=0.1).cuda()

    n_params = sum(p.numel() for p in backbone.parameters())
    print(f'  Backbone IResNet50 参数量: {n_params/1e6:.2f} M')

    # ---------------- 优化器 / 调度 ----------------
    optimizer = torch.optim.SGD(
        [{'params': backbone.parameters()},
         {'params': arcface.parameters()}],
        lr=LR0, momentum=0.9, weight_decay=5e-4)
    scaler = GradScaler('cuda')

    start_epoch = 0
    if RESUME and os.path.exists(RESUME):
        ckpt = torch.load(RESUME, map_location='cpu', weights_only=False)
        backbone.load_state_dict(ckpt['backbone'])
        arcface.load_state_dict(ckpt['arcface'])
        optimizer.load_state_dict(ckpt['optimizer'])
        start_epoch = ckpt['epoch'] + 1
        print(f'  [resume] 从 {RESUME} 续训, 从 epoch {start_epoch} 开始')

    def lr_at(epoch):
        if epoch < WARMUP:
            return LR0 * (epoch + 1) / WARMUP
        progress = (epoch - WARMUP) / max(1, EPOCHS - WARMUP)
        return LR0 * 0.01 + 0.5 * (LR0 - LR0 * 0.01) * (1 + math.cos(math.pi * progress))

    # ---------------- 训练循环 ----------------
    print(f'\n  {"Epoch":>6s} | {"lr":>9s} | {"loss":>8s} | {"acc":>7s} | {"time":>8s} | {"剩余":>8s}')
    print('  ' + '-' * 60)
    t_start = time.time()

    for epoch in range(start_epoch, EPOCHS):
        lr = lr_at(epoch)
        for g in optimizer.param_groups:
            g['lr'] = lr

        backbone.train()
        arcface.train()
        run_loss = 0.0
        run_correct = 0
        run_total = 0
        t_ep = time.time()

        for i, (imgs, labels) in enumerate(loader):
            if MAX_STEPS and i >= MAX_STEPS:
                break
            imgs = imgs.cuda(non_blocking=True)
            labels = labels.cuda(non_blocking=True)

            optimizer.zero_grad()
            with autocast('cuda'):
                feats = backbone(imgs)
                logits = arcface(feats, labels)
                loss = ce_loss(logits, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            run_loss += loss.item() * imgs.size(0)
            run_correct += (logits.argmax(dim=1) == labels).sum().item()
            run_total += imgs.size(0)

        n_it = max(1, i + 1)
        avg_loss = run_loss / run_total
        acc = run_correct / run_total
        el = time.time() - t_ep
        total_el = time.time() - t_start
        remain = total_el / (epoch - start_epoch + 1) * (EPOCHS - epoch - 1)
        print(f'  {epoch+1:6d} | {lr:9.6f} | {avg_loss:8.4f} | {acc:7.4f} | {el:7.1f}s | {remain/3600:6.1f}h')

        # 存档 (每 epoch 存 last.pt, 每 5 epoch 存 epoch_N.pt)
        ckpt = {
            'backbone': backbone.state_dict(),
            'arcface': arcface.state_dict(),
            'optimizer': optimizer.state_dict(),
            'epoch': epoch,
            'num_classes': num_classes,
        }
        torch.save(ckpt, os.path.join(OUT_DIR, 'last.pt'))
        if (epoch + 1) % 5 == 0:
            torch.save(ckpt, os.path.join(OUT_DIR, f'epoch_{epoch+1}.pt'))

    print('=' * 64)
    print(f'  Training Complete!  End: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
    print(f'  模型保存在: {OUT_DIR}/last.pt')
    print('=' * 64)


if __name__ == '__main__':
    main()


In [ ]:
# ① 转换 .rec → ImageFolder (全量 ~49 万张, 约 20-40 分钟)
!python kaggle_convert.py

# 冒烟: 只转前 2000 张, 快速验证路径/格式 (通过后再跑上面全量)
# !python kaggle_convert.py --limit 2000

In [ ]:
# ② 冒烟训练: 1 epoch 只跑 3 个 batch, 确认身份数≈10572、loss 能下降
!python kaggle_train.py --epochs 1 --batch 64 --max-steps 3

# ③ 正式训练 (20 epoch): 冒烟通过后注释掉上面、取消下面这行
# !python kaggle_train.py --epochs 20 --batch 256